# Laboratorium terbuka: reaksi kimia nonlinear

Notebook ini merupakan pendamping komputasi mandiri untuk Bab 8. Seluruh perhitungan memakai NumPy, SciPy, dan Matplotlib, dapat dijalankan secara luring, dan tidak membaca data tersembunyi maupun mengakses jaringan. Notebook ini bukan port kode PPLANE, MAPLE, MATHEMATICA, atau perangkat lunak berpemilik lain.

Eksperimen memeriksa hukum aksi massa dan konservasi pada Soal 1, bifurkasi Hopf serta perubahan fokus–simpul Brusselator, kestabilan dan siklus limit Oregonator tereduksi, dan keluarga orbit periodik Lotka–Volterra pada Soal 12–13.

**ID unit:** O005-LEGA-V101-CH08  
**ID notebook:** O005-LEGA-V101-CH08-NB01  
**Lisensi notebook:** CC BY-NC-SA 4.0  
**Provenans:** pendamping komputasi baru yang diturunkan dari persamaan dan soal Bab 8; tidak ada kode aplikasi sumber yang disalin.  
**Catatan perubahan:** alur kerja perangkat lunak berpemilik pada Soal 6, 7, 10, dan 11 diganti dengan analisis terbuka yang dapat diuji, integrasi SciPy deterministik, serta diagnostik numerik yang tidak bergantung pada perbandingan piksel gambar.

## Batas model dan pilihan yang dinyatakan

- Untuk Soal 1, setiap arah reaksi reversibel diperlakukan sebagai reaksi elementer tersendiri. Vektor konservasi dibuktikan dari matriks stoikiometri dan diperiksa lagi sepanjang solusi numerik.
- Brusselator digunakan dalam bentuk Persamaan (8.1), dengan $A>0$ dan $B>0$. Pada $A=1$, Hopf terjadi di $B=2$, sedangkan $B=4$ adalah batas diskriminan lokal antara fokus dan simpul tak stabil.
- Oregonator direduksi dengan cabang positif $h_q(y)$ yang tercetak dalam bab. Analisis di sini khusus untuk $f=s=1$ dan $\omega=2$; solver LSODA dipilih karena sesuai untuk dinamika dengan skala waktu yang berbeda.
- Semua keadaan awal, toleransi, kisi waktu, dan parameter dinyatakan di dalam sel. Tidak ada bilangan acak, berkas eksternal, permintaan jaringan, atau keadaan sesi tersembunyi.

In [ ]:
import platform
import time

import numpy as np
import scipy
from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar
import matplotlib
import matplotlib.pyplot as plt

np.set_printoptions(precision=9, suppress=True)

def selesaikan(fun, rentang, awal, args=(), method="DOP853", t_eval=None, max_step=np.inf, events=None):
    awal = np.asarray(awal, dtype=float)
    hasil = solve_ivp(
        fun,
        rentang,
        awal,
        args=args,
        method=method,
        rtol=1e-9,
        atol=1e-11,
        t_eval=t_eval,
        max_step=max_step,
        events=events,
    )
    assert hasil.success, hasil.message
    assert hasil.y.shape[0] == awal.size
    assert np.all(np.isfinite(hasil.t)) and np.all(np.isfinite(hasil.y))
    assert hasil.t.size > 1 and np.all(np.diff(hasil.t) > 0.0)
    return hasil

def akhiri_gambar(fig):
    fig.canvas.draw()
    if matplotlib.get_backend().lower() == "agg":
        plt.close(fig)
    else:
        plt.show()

assert isinstance(matplotlib.get_backend(), str) and matplotlib.get_backend()
assert tuple(int(bagian) for bagian in np.__version__.split(".")[:2]) >= (2, 0)
assert tuple(int(bagian) for bagian in scipy.__version__.split(".")[:2]) >= (1, 10)

waktu_mulai_notebook = time.perf_counter()
print(f"Python/NumPy/SciPy/Matplotlib: {platform.python_version()} / {np.__version__} / {scipy.__version__} / {matplotlib.__version__}")
print("Backend Matplotlib:", matplotlib.get_backend())

## 1. Soal 1: aksi massa dan hukum konservasi

Untuk jaringan

$$A+B \rightleftharpoons C \longrightarrow B+D,\qquad B+E \rightleftharpoons F,$$

kita melacak $z=([A],[B],[C],[E],[F])$. Dengan laju elementer

$$r=(k_1AB,k_2C,k_3C,k_4BE,k_5F)^T,$$

persamaan laju adalah $\dot z=Sr$. Karena setiap bentuk bebas atau terikat membawa satu unit gugus $B$, jumlah $[B]+[C]+[F]$ harus konstan. Jaringan juga mempertahankan $[E]+[F]$.

In [ ]:
S_P1 = np.array([
    [-1,  1,  0,  0,  0],  # A
    [-1,  1,  1, -1,  1],  # B
    [ 1, -1, -1,  0,  0],  # C
    [ 0,  0,  0, -1,  1],  # E
    [ 0,  0,  0,  1, -1],  # F
], dtype=float)

def laju_elementer_P1(z, k):
    A, B, C, E, F = np.asarray(z, dtype=float)
    k1, k2, k3, k4, k5 = k
    return np.array([k1 * A * B, k2 * C, k3 * C, k4 * B * E, k5 * F])

def rhs_P1(t, z, *k):
    return S_P1 @ laju_elementer_P1(z, k)

konservasi_B = np.array([0.0, 1.0, 1.0, 0.0, 1.0])
konservasi_E = np.array([0.0, 0.0, 0.0, 1.0, 1.0])
assert S_P1.shape == (5, 5)
assert np.array_equal(konservasi_B @ S_P1, np.zeros(5))
assert np.array_equal(konservasi_E @ S_P1, np.zeros(5))

k_P1 = (1.2, 0.4, 0.3, 0.7, 0.2)
z_uji_P1 = np.array([1.1, 0.8, 0.15, 0.6, 0.1])
r_uji_P1 = laju_elementer_P1(z_uji_P1, k_P1)
rhs_uji_P1 = rhs_P1(0.0, z_uji_P1, *k_P1)
A, B, C, E, F = z_uji_P1
k1, k2, k3, k4, k5 = k_P1
rhs_eksplisit_P1 = np.array([
    -k1 * A * B + k2 * C,
    -k1 * A * B + (k2 + k3) * C - k4 * B * E + k5 * F,
    k1 * A * B - (k2 + k3) * C,
    -k4 * B * E + k5 * F,
    k4 * B * E - k5 * F,
])
assert np.all(r_uji_P1 >= 0.0)
assert np.allclose(rhs_uji_P1, rhs_eksplisit_P1)
assert np.isclose(konservasi_B @ rhs_uji_P1, 0.0)
assert np.isclose(konservasi_E @ rhs_uji_P1, 0.0)
for indeks in range(5):
    z_muka = z_uji_P1.copy()
    z_muka[indeks] = 0.0
    assert rhs_P1(0.0, z_muka, *k_P1)[indeks] >= -1e-15

awal_P1 = np.array([1.1, 0.8, 0.15, 0.6, 0.1])
hasil_P1 = selesaikan(
    rhs_P1, (0.0, 45.0), awal_P1, k_P1,
    t_eval=np.linspace(0.0, 45.0, 901), max_step=0.08,
)
assert np.min(hasil_P1.y) >= -2e-11
jumlah_B = konservasi_B @ hasil_P1.y
jumlah_E = konservasi_E @ hasil_P1.y
drift_B = float(np.max(np.abs(jumlah_B - jumlah_B[0])))
drift_E = float(np.max(np.abs(jumlah_E - jumlah_E[0])))
assert drift_B < 2e-10
assert drift_E < 2e-10
assert np.isclose(jumlah_B[-1], awal_P1[1] + awal_P1[2] + awal_P1[4], atol=2e-10)
assert np.isclose(jumlah_E[-1], awal_P1[3] + awal_P1[4], atol=2e-10)

fig, ax = plt.subplots(figsize=(8.6, 4.8), constrained_layout=True)
for indeks, nama in enumerate(("[A]", "[B]", "[C]", "[E]", "[F]")):
    ax.plot(hasil_P1.t, hasil_P1.y[indeks], label=nama)
ax.set(xlabel="waktu", ylabel="konsentrasi", title="Soal 1: dinamika aksi massa")
ax.grid(alpha=0.2)
ax.legend(ncol=3)
akhiri_gambar(fig)

print(f"Drift [B]+[C]+[F] = {drift_B:.3e}; drift [E]+[F] = {drift_E:.3e}")
print("Keadaan akhir P1:", hasil_P1.y[:, -1])

## 2. Brusselator: titik tetap, Hopf, dan batas fokus–simpul

Model umum Persamaan (8.1) adalah

$$\dot X=A-(B+1)X+X^2Y,\qquad \dot Y=BX-X^2Y.$$

Untuk $A>0$, satu-satunya titik tetap positif ialah $P=(A,B/A)$. Matriks Jacobi di sana, jejak, determinan, dan diskriminannya adalah

$$J(P)=\begin{pmatrix}B-1&A^2\\-B&-A^2\end{pmatrix},\quad T=B-1-A^2,\quad D=A^2,\quad \Delta=T^2-4D.$$

Karena $D>0$, perubahan kestabilan terjadi ketika $T=0$, yaitu $B_H=1+A^2$. Untuk $A=1$, $B_H=2$. Pada cabang tak stabil, $\Delta<0$ untuk $2<B<4$ (fokus) dan $\Delta>0$ untuk $B>4$ (simpul); tepat di $B=4$ terdapat nilai eigen ganda $+1$.

In [ ]:
def brusselator(t, z, A, B):
    X, Y = z
    return np.array([A - (B + 1.0) * X + X * X * Y, B * X - X * X * Y])

def titik_brusselator(A, B):
    return np.array([A, B / A])

def jacobian_brusselator(X, Y, A, B):
    return np.array([[-(B + 1.0) + 2.0 * X * Y, X * X], [B - 2.0 * X * Y, -X * X]])

def ringkasan_linear_brusselator(A, B):
    P = titik_brusselator(A, B)
    J = jacobian_brusselator(*P, A, B)
    jejak = float(np.trace(J))
    determinan = float(np.linalg.det(J))
    diskriminan = jejak * jejak - 4.0 * determinan
    return P, J, jejak, determinan, diskriminan, np.linalg.eigvals(J)

for A_uji, B_uji in ((0.7, 0.8), (1.0, 3.0), (1.8, 5.0)):
    P_uji, J_uji, T_uji, D_uji, Delta_uji, eig_uji = ringkasan_linear_brusselator(A_uji, B_uji)
    assert np.linalg.norm(brusselator(0.0, P_uji, A_uji, B_uji), ord=np.inf) < 1e-13
    assert np.isclose(T_uji, B_uji - 1.0 - A_uji**2)
    assert np.isclose(D_uji, A_uji**2) and D_uji > 0.0
    assert np.isclose(np.prod(eig_uji), D_uji)

P_fd = titik_brusselator(1.3, 2.7)
eps_fd = 1e-6
J_fd = np.column_stack([
    (brusselator(0.0, P_fd + eps_fd * np.eye(2)[j], 1.3, 2.7)
     - brusselator(0.0, P_fd - eps_fd * np.eye(2)[j], 1.3, 2.7)) / (2.0 * eps_fd)
    for j in range(2)
])
assert np.allclose(J_fd, jacobian_brusselator(*P_fd, 1.3, 2.7), rtol=2e-9, atol=2e-9)

for A_uji in (0.5, 1.0, 2.0):
    B_H = 1.0 + A_uji**2
    _, _, T_H, D_H, _, eig_H = ringkasan_linear_brusselator(A_uji, B_H)
    assert np.isclose(T_H, 0.0, atol=1e-13)
    assert D_H > 0.0 and np.allclose(eig_H.real, 0.0, atol=2e-8)
    assert np.allclose(np.sort(np.abs(eig_H.imag)), [A_uji, A_uji])

ringkasan_A1 = {B: ringkasan_linear_brusselator(1.0, B) for B in (1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0)}
assert ringkasan_A1[1.5][2] < 0.0 and ringkasan_A1[1.5][4] < 0.0
assert np.isclose(ringkasan_A1[2.0][2], 0.0) and np.isclose(ringkasan_A1[2.0][3], 1.0)
assert np.allclose(np.sort(np.abs(ringkasan_A1[2.0][5].imag)), [1.0, 1.0])
assert ringkasan_A1[2.5][2] > 0.0 and ringkasan_A1[2.5][4] < 0.0
assert ringkasan_A1[3.0][4] < 0.0
assert np.isclose(ringkasan_A1[4.0][4], 0.0, atol=1e-13)
assert np.allclose(ringkasan_A1[4.0][5], [1.0, 1.0], atol=2e-8)
assert np.linalg.matrix_rank(ringkasan_A1[4.0][1] - np.eye(2)) == 1
assert ringkasan_A1[5.0][4] > 0.0 and np.all(ringkasan_A1[5.0][5].real > 0.0)
assert ringkasan_A1[6.0][4] > 0.0 and np.all(np.isreal(ringkasan_A1[6.0][5]))

print("Hopf Brusselator untuk A=1: B_H=2")
print("B=4: Delta=", ringkasan_A1[4.0][4], "; eigen=", ringkasan_A1[4.0][5])

## 3. Brusselator numerik untuk $B=1{,}5$ sampai $6$

Semua lintasan memakai $A=1$ dan gangguan kecil yang sama relatif terhadap $P=(1,B)$. Jendela akhir menguji apakah amplitudo menghilang atau menetap. Untuk $B>2$, perpotongan naik $X=1$ menjadi penampang Poincaré: periode dan koordinat kembalinya harus berulang, bukan sekadar tampak tertutup pada plot.

In [ ]:
def potongan_X_naik(t, z, A, B):
    return z[0] - A

potongan_X_naik.direction = 1
potongan_X_naik.terminal = False

B_daftar = np.array([1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0])
kisi_brusselator = np.linspace(0.0, 120.0, 3001)
hasil_brusselator = {}
for B_nilai in B_daftar:
    awal = np.array([1.2, B_nilai + 0.15])
    hasil = selesaikan(
        brusselator, (0.0, 120.0), awal, (1.0, float(B_nilai)),
        method="DOP853", t_eval=kisi_brusselator, max_step=0.12, events=potongan_X_naik,
    )
    assert np.min(hasil.y) >= -2e-10
    assert np.linalg.norm(brusselator(0.0, titik_brusselator(1.0, B_nilai), 1.0, B_nilai)) < 1e-13
    hasil_brusselator[float(B_nilai)] = hasil

amplitudo_akhir = []
amplitudo_sebelum = []
for B_nilai in B_daftar:
    hasil = hasil_brusselator[float(B_nilai)]
    amplitudo_sebelum.append(np.ptp(hasil.y[:, (hasil.t >= 60.0) & (hasil.t < 90.0)], axis=1))
    amplitudo_akhir.append(np.ptp(hasil.y[:, hasil.t >= 90.0], axis=1))
amplitudo_sebelum = np.asarray(amplitudo_sebelum)
amplitudo_akhir = np.asarray(amplitudo_akhir)
assert amplitudo_akhir.shape == (7, 2)
assert amplitudo_akhir[0, 0] < 1e-7 and amplitudo_akhir[0, 1] < 1e-7
assert 0.1 < amplitudo_akhir[1, 0] < 0.5
assert np.all(amplitudo_akhir[2:, 0] > 1.0)
assert np.all(np.diff(amplitudo_akhir[2:, 0]) > 0.5)
assert np.max(np.abs(amplitudo_akhir[2:] - amplitudo_sebelum[2:]) / amplitudo_akhir[2:]) < 0.025

hasil_B2 = hasil_brusselator[2.0]
jarak_B2 = np.linalg.norm(hasil_B2.y - titik_brusselator(1.0, 2.0)[:, None], axis=0)
rata_awal_B2 = float(np.mean(jarak_B2[(hasil_B2.t >= 30.0) & (hasil_B2.t < 60.0)]))
rata_akhir_B2 = float(np.mean(jarak_B2[hasil_B2.t >= 90.0]))
assert rata_akhir_B2 < 0.8 * rata_awal_B2

diagnostik_kembali = {}
for B_nilai in B_daftar[B_daftar > 2.0]:
    hasil = hasil_brusselator[float(B_nilai)]
    t_potong = hasil.t_events[0]
    y_potong = hasil.y_events[0]
    mask_akhir = t_potong > 70.0
    assert np.count_nonzero(mask_akhir) >= 4
    periode = np.diff(t_potong[mask_akhir])[-3:]
    koordinat_kembali = y_potong[mask_akhir][-4:, 1]
    assert np.ptp(periode) < 1e-5
    assert np.ptp(koordinat_kembali) < 1e-5
    diagnostik_kembali[float(B_nilai)] = (float(np.mean(periode)), float(np.ptp(koordinat_kembali)))

hasil_B6 = hasil_brusselator[6.0]
mask_B6 = hasil_B6.t >= 80.0
X_B6, Y_B6 = hasil_B6.y[:, mask_B6]
rhs_B6 = np.column_stack([brusselator(0.0, z, 1.0, 6.0) for z in hasil_B6.y[:, mask_B6].T])
fraksi_lambat_B6 = float(np.mean(X_B6 < 1.0))
rasio_kecepatan_B6 = float(np.max(np.abs(rhs_B6[0])) / np.median(np.abs(rhs_B6[0])))
assert np.ptp(X_B6) > 12.0
assert np.ptp(Y_B6) > 12.0
assert fraksi_lambat_B6 > 0.8
assert rasio_kecepatan_B6 > 1000.0

fig, sumbu = plt.subplots(7, 2, figsize=(11.8, 20.0), constrained_layout=True)
for baris, B_nilai in enumerate(B_daftar):
    hasil = hasil_brusselator[float(B_nilai)]
    P = titik_brusselator(1.0, B_nilai)
    sumbu[baris, 0].plot(hasil.y[0], hasil.y[1], color="#0072B2", linewidth=1.0)
    sumbu[baris, 0].plot(*P, "o", color="#D55E00", markersize=4)
    sumbu[baris, 0].set(xlabel="X", ylabel="Y", title=f"Bidang fase, B={B_nilai:g}")
    mask_waktu = hasil.t >= 90.0
    sumbu[baris, 1].plot(hasil.t[mask_waktu], hasil.y[0, mask_waktu], label="X")
    sumbu[baris, 1].plot(hasil.t[mask_waktu], hasil.y[1, mask_waktu], label="Y")
    sumbu[baris, 1].set(xlabel="t", ylabel="konsentrasi", title=f"Deret waktu akhir, B={B_nilai:g}")
    for ax in sumbu[baris]:
        ax.grid(alpha=0.18)
    sumbu[baris, 1].legend(fontsize=8, ncol=2)
akhiri_gambar(fig)

print("Amplitudo akhir Brusselator [rentang X, rentang Y]:")
for B_nilai, amp in zip(B_daftar, amplitudo_akhir):
    print(f"  B={B_nilai:g}: {amp[0]:.8f}, {amp[1]:.8f}")
print("Periode Poincare B>2:", {B: nilai[0] for B, nilai in diagnostik_kembali.items()})
print(f"B=6: fraksi X<1 = {fraksi_lambat_B6:.4f}; rasio laju cepat/laju tipikal = {rasio_kecepatan_B6:.1f}")

## 4. Oregonator tereduksi: titik tetap eksak dan ambang Hopf

Dengan

$$h_q(y)=\frac{1-y+\sqrt{(1-y)^2+4qy}}{2q},\qquad f=s=1,\qquad \omega=2,$$

sistem tereduksi menjadi

$$\dot y=-y-yh_q(y)+z,\qquad \dot z=2\bigl(h_q(y)-z\bigr).$$

Pada titik tetap positif, $z_*=x_*=h_q(y_*)$ dan $y_*=x_*/(1+x_*)$. Substitusi ke persamaan kuadrat untuk $h_q$ memberi rumus eksak

$$x_*=z_*=\frac{\sqrt{1+8/q}-1}{2},\qquad y_*=\frac{\sqrt{1+8/q}-1}{\sqrt{1+8/q}+1}.$$

Dari diferensiasi implisit, $h'_q(y)=(1-h_q(y))/\sqrt{(1-y)^2+4qy}$. Maka

$$J=\begin{pmatrix}-1-h-yh'&1\\2h'&-2\end{pmatrix},\quad T=-3-h-yh',\quad D=2\bigl(1+h+(y-1)h'\bigr),$$

dengan $h$ dan $h'$ dievaluasi di $y_*$.

In [ ]:
def h_q(y, q):
    y = np.asarray(y, dtype=float)
    diskriminan = (1.0 - y)**2 + 4.0 * q * y
    return (1.0 - y + np.sqrt(diskriminan)) / (2.0 * q)

def turunan_h_q(y, q):
    h = h_q(y, q)
    akar = np.sqrt((1.0 - y)**2 + 4.0 * q * y)
    return (1.0 - h) / akar

def titik_oregonator(q):
    akar = np.sqrt(1.0 + 8.0 / q)
    x = 0.5 * (akar - 1.0)
    y = (akar - 1.0) / (akar + 1.0)
    return np.array([y, x])

def oregonator_tereduksi(t, z, q):
    y, konsentrasi_z = z
    x = h_q(y, q)
    return np.array([-y - y * x + konsentrasi_z, 2.0 * (x - konsentrasi_z)])

def jacobian_oregonator(y, konsentrasi_z, q):
    h = h_q(y, q)
    hp = turunan_h_q(y, q)
    return np.array([[-1.0 - h - y * hp, 1.0], [2.0 * hp, -2.0]])

def ringkasan_linear_oregonator(q):
    P = titik_oregonator(q)
    J = jacobian_oregonator(*P, q)
    y, _ = P
    h = h_q(y, q)
    hp = turunan_h_q(y, q)
    jejak = float(-3.0 - h - y * hp)
    determinan = float(2.0 * (1.0 + h + (y - 1.0) * hp))
    return P, J, jejak, determinan, np.linalg.eigvals(J)

for q_uji in (0.01, 0.03, 0.1):
    for y_uji in (0.0, 0.2, 1.0, 5.0):
        h_uji = h_q(y_uji, q_uji)
        assert np.isfinite(h_uji) and h_uji > 0.0
        assert abs(q_uji * h_uji**2 + (y_uji - 1.0) * h_uji - y_uji) < 2e-11
    assert np.isclose(h_q(0.0, q_uji), 1.0 / q_uji)
    assert np.isclose(h_q(1.0, q_uji), 1.0 / np.sqrt(q_uji))

y_fd, q_fd = 0.83, 0.06
hp_fd = (h_q(y_fd + 1e-6, q_fd) - h_q(y_fd - 1e-6, q_fd)) / 2e-6
assert np.isclose(hp_fd, turunan_h_q(y_fd, q_fd), rtol=2e-8, atol=2e-8)

for q_uji in (0.01, 0.054, 0.1):
    P_uji, J_uji, T_uji, D_uji, eig_uji = ringkasan_linear_oregonator(q_uji)
    y_uji, z_uji = P_uji
    x_uji = h_q(y_uji, q_uji)
    assert y_uji > 0.0 and z_uji > 0.0
    assert np.isclose(z_uji, x_uji, rtol=2e-13)
    assert np.isclose(y_uji, x_uji / (1.0 + x_uji), rtol=2e-13)
    assert np.linalg.norm(oregonator_tereduksi(0.0, P_uji, q_uji), ord=np.inf) < 2e-11
    assert np.isclose(T_uji, np.trace(J_uji)) and np.isclose(D_uji, np.linalg.det(J_uji))
    assert D_uji > 0.0

P_fd_ore = titik_oregonator(0.07)
J_fd_ore = np.column_stack([
    (oregonator_tereduksi(0.0, P_fd_ore + 1e-6 * np.eye(2)[j], 0.07)
     - oregonator_tereduksi(0.0, P_fd_ore - 1e-6 * np.eye(2)[j], 0.07)) / 2e-6
    for j in range(2)
])
assert np.allclose(J_fd_ore, jacobian_oregonator(*P_fd_ore, 0.07), rtol=3e-8, atol=3e-8)

P_q01, J_q01, T_q01, D_q01, eig_q01 = ringkasan_linear_oregonator(0.1)
P_q001, J_q001, T_q001, D_q001, eig_q001 = ringkasan_linear_oregonator(0.01)
assert T_q01 < 0.0 and D_q01 > 0.0
assert np.all(eig_q01.real < 0.0) and np.all(np.abs(eig_q01.imag) > 0.0)
assert T_q001 > 0.0 and D_q001 > 0.0
assert np.all(eig_q001.real > 0.0) and np.allclose(eig_q001.imag, 0.0)

akar_hopf = root_scalar(
    lambda q: ringkasan_linear_oregonator(q)[2],
    bracket=(0.01, 0.1), method="brentq", xtol=1e-14, rtol=1e-14,
)
q_H = float(akar_hopf.root)
P_H, J_H, T_H, D_H, eig_H = ringkasan_linear_oregonator(q_H)
assert akar_hopf.converged
assert abs(q_H - 0.0540135346) < 5e-11
assert abs(T_H) < 1e-10
assert D_H > 0.0 and np.allclose(eig_H.real, 0.0, atol=1e-9)
assert np.all(np.abs(eig_H.imag) > 4.0)

print(f"q_H Oregonator = {q_H:.12f}; trace={T_H:.3e}; det={D_H:.9f}")
print("q=0,1: titik tetap/eigen =", P_q01, eig_q01)
print("q=0,01: titik tetap/eigen =", P_q001, eig_q001)

## 5. Oregonator numerik: fokus stabil dan siklus limit

Kasus $q=0{,}1$ diuji sebagai fokus stabil. Kasus $q=0{,}01$ memakai LSODA dengan toleransi ketat dan penampang Poincaré $y=y_*$ berarah naik. Kesamaan amplitudo pada dua jendela akhir, periode kembali, dan koordinat kembali menguji siklus limit menarik tanpa membandingkan grafik secara eksak.

In [ ]:
def potongan_y_oregonator(t, z, q):
    return z[0] - titik_oregonator(q)[0]

potongan_y_oregonator.direction = 1
potongan_y_oregonator.terminal = False

awal_stabil = titik_oregonator(0.1) + np.array([0.15, -0.5])
hasil_ore_stabil = selesaikan(
    oregonator_tereduksi, (0.0, 40.0), awal_stabil, (0.1,),
    method="LSODA", t_eval=np.linspace(0.0, 40.0, 1601), max_step=0.1,
    events=potongan_y_oregonator,
)
hasil_ore_siklus = selesaikan(
    oregonator_tereduksi, (0.0, 100.0), np.array([0.8, 3.0]), (0.01,),
    method="LSODA", t_eval=np.linspace(0.0, 100.0, 4001), max_step=0.1,
    events=potongan_y_oregonator,
)
assert np.min(hasil_ore_stabil.y) >= -2e-10
assert np.min(hasil_ore_siklus.y) >= -2e-10
galat_stabil = float(np.linalg.norm(hasil_ore_stabil.y[:, -1] - titik_oregonator(0.1)))
amp_stabil = np.ptp(hasil_ore_stabil.y[:, hasil_ore_stabil.t >= 30.0], axis=1)
assert galat_stabil < 1e-8
assert np.max(amp_stabil) < 1e-7

mask_60_80 = (hasil_ore_siklus.t >= 60.0) & (hasil_ore_siklus.t < 80.0)
mask_80_100 = hasil_ore_siklus.t >= 80.0
amp_60_80 = np.ptp(hasil_ore_siklus.y[:, mask_60_80], axis=1)
amp_80_100 = np.ptp(hasil_ore_siklus.y[:, mask_80_100], axis=1)
assert amp_80_100[0] > 5.0 and amp_80_100[1] > 25.0
assert np.max(np.abs(amp_80_100 - amp_60_80) / amp_80_100) < 1e-3
assert np.linalg.norm(hasil_ore_siklus.y[:, -1] - titik_oregonator(0.01)) > 1.0

t_kembali_ore = hasil_ore_siklus.t_events[0]
z_kembali_ore = hasil_ore_siklus.y_events[0]
mask_kembali_ore = t_kembali_ore > 60.0
assert np.count_nonzero(mask_kembali_ore) >= 10
periode_ore = np.diff(t_kembali_ore[mask_kembali_ore])[-8:]
bagian_z_ore = z_kembali_ore[mask_kembali_ore][-9:, 1]
assert np.ptp(periode_ore) < 1e-5
assert np.ptp(bagian_z_ore) < 1e-4
assert 2.0 < np.mean(periode_ore) < 2.3

fig, sumbu = plt.subplots(2, 2, figsize=(11.2, 8.0), constrained_layout=True)
sumbu[0, 0].plot(hasil_ore_stabil.y[0], hasil_ore_stabil.y[1], color="#0072B2")
sumbu[0, 0].plot(*titik_oregonator(0.1), "o", color="#D55E00")
sumbu[0, 0].set(xlabel="y", ylabel="z", title="q=0,1: fokus stabil")
sumbu[0, 1].plot(hasil_ore_stabil.t, hasil_ore_stabil.y[0], label="y")
sumbu[0, 1].plot(hasil_ore_stabil.t, hasil_ore_stabil.y[1], label="z")
sumbu[0, 1].set(xlabel="tau", ylabel="keadaan", title="Konvergensi ke titik tetap")
mask_fase = hasil_ore_siklus.t >= 20.0
sumbu[1, 0].plot(hasil_ore_siklus.y[0, mask_fase], hasil_ore_siklus.y[1, mask_fase], color="#009E73")
sumbu[1, 0].plot(*titik_oregonator(0.01), "o", color="#D55E00")
sumbu[1, 0].set(xlabel="y", ylabel="z", title="q=0,01: siklus limit menarik")
mask_waktu_ore = hasil_ore_siklus.t >= 85.0
sumbu[1, 1].plot(hasil_ore_siklus.t[mask_waktu_ore], hasil_ore_siklus.y[0, mask_waktu_ore], label="y")
sumbu[1, 1].plot(hasil_ore_siklus.t[mask_waktu_ore], hasil_ore_siklus.y[1, mask_waktu_ore], label="z")
sumbu[1, 1].set(xlabel="tau", ylabel="keadaan", title="Osilasi menetap")
for ax in sumbu.flat:
    ax.grid(alpha=0.18)
sumbu[0, 1].legend()
sumbu[1, 1].legend()
akhiri_gambar(fig)

print(f"q=0,1: galat akhir ke fokus = {galat_stabil:.3e}")
print(f"q=0,01: amplitudo akhir y/z = {amp_80_100}; periode = {np.mean(periode_ore):.9f}")
print(f"Sebaran koordinat kembali z = {np.ptp(bagian_z_ore):.3e}")

## 6. Soal 12–13: Lotka–Volterra dan diagnostik gangguan

Jika $[A]$ dipertahankan konstan, jaringan Soal 12 memberi

$$\dot X=X(\alpha-\beta Y),\qquad \dot Y=Y(\beta X-\gamma),$$

dengan $\alpha=k_1[A]$, $\beta=k_2$, dan $\gamma=k_3$. Ini adalah Lotka–Volterra klasik. Berbeda dari sebuah siklus limit menarik, setiap syarat awal positif mempertahankan tingkat invariannya sendiri,

$$H(X,Y)=\beta X-\gamma\log X+\beta Y-\alpha\log Y.$$

Dua gangguan dengan nilai $H$ berbeda karena itu tetap pada dua orbit periodik berbeda; keduanya tidak dapat menyusut menuju kurva tertutup yang sama.

In [ ]:
def lotka_volterra_kimia(t, z, alpha, beta, gamma):
    X, Y = z
    return np.array([X * (alpha - beta * Y), Y * (beta * X - gamma)])

def invarian_LV(z, alpha, beta, gamma):
    X, Y = np.asarray(z)
    return beta * X - gamma * np.log(X) + beta * Y - alpha * np.log(Y)

k1_LV, A_LV, k2_LV, k3_LV = 1.1, 0.9, 0.8, 0.7
alpha_LV, beta_LV, gamma_LV = k1_LV * A_LV, k2_LV, k3_LV
P_LV = np.array([gamma_LV / beta_LV, alpha_LV / beta_LV])
assert np.linalg.norm(lotka_volterra_kimia(0.0, P_LV, alpha_LV, beta_LV, gamma_LV)) < 1e-14
assert lotka_volterra_kimia(0.0, [0.0, 1.0], alpha_LV, beta_LV, gamma_LV)[0] == 0.0
assert lotka_volterra_kimia(0.0, [1.0, 0.0], alpha_LV, beta_LV, gamma_LV)[1] == 0.0

z_identitas = np.array([1.2, 0.9])
grad_H = np.array([beta_LV - gamma_LV / z_identitas[0], beta_LV - alpha_LV / z_identitas[1]])
assert np.isclose(grad_H @ lotka_volterra_kimia(0.0, z_identitas, alpha_LV, beta_LV, gamma_LV), 0.0)
J_LV = np.array([[0.0, -beta_LV * P_LV[0]], [beta_LV * P_LV[1], 0.0]])
eig_LV = np.linalg.eigvals(J_LV)
assert np.allclose(eig_LV.real, 0.0) and np.all(np.abs(eig_LV.imag) > 0.0)

awal_LV = (P_LV * np.array([1.15, 0.85]), P_LV * np.array([1.35, 0.65]))
hasil_LV = [
    selesaikan(
        lotka_volterra_kimia, (0.0, 80.0), awal, (alpha_LV, beta_LV, gamma_LV),
        method="DOP853", t_eval=np.linspace(0.0, 80.0, 2001), max_step=0.1,
    )
    for awal in awal_LV
]
nilai_H = []
amplitudo_LV = []
for hasil in hasil_LV:
    assert np.min(hasil.y) > 0.0
    H = invarian_LV(hasil.y, alpha_LV, beta_LV, gamma_LV)
    drift_H = float(np.max(np.abs(H - H[0])))
    assert drift_H < 2e-10
    nilai_H.append(float(H[0]))
    amp_tengah = np.ptp(hasil.y[:, (hasil.t >= 20.0) & (hasil.t < 40.0)], axis=1)
    amp_akhir = np.ptp(hasil.y[:, hasil.t >= 60.0], axis=1)
    assert np.max(np.abs(amp_akhir - amp_tengah) / amp_akhir) < 0.02
    amplitudo_LV.append(amp_akhir)
nilai_H = np.asarray(nilai_H)
amplitudo_LV = np.asarray(amplitudo_LV)
assert abs(nilai_H[1] - nilai_H[0]) > 0.05
assert np.all(amplitudo_LV[1] > 2.0 * amplitudo_LV[0])
assert np.linalg.norm(hasil_LV[0].y[:, -1] - P_LV) > 0.1
assert np.linalg.norm(hasil_LV[1].y[:, -1] - P_LV) > 0.1

fig, sumbu = plt.subplots(1, 2, figsize=(11.0, 4.6), constrained_layout=True)
for indeks, hasil in enumerate(hasil_LV, start=1):
    sumbu[0].plot(hasil.y[0], hasil.y[1], label=f"gangguan {indeks}")
    sumbu[1].plot(hasil.t, hasil.y[0], label=f"X, gangguan {indeks}")
sumbu[0].plot(*P_LV, "o", color="#D55E00", label="titik tetap")
sumbu[0].set(xlabel="X", ylabel="Y", title="Dua tingkat invarian, dua orbit")
sumbu[1].set(xlabel="waktu", ylabel="X", title="Amplitudo tidak meluruh")
for ax in sumbu:
    ax.grid(alpha=0.18)
    ax.legend()
akhiri_gambar(fig)

waktu_notebook = time.perf_counter() - waktu_mulai_notebook
assert waktu_notebook < 60.0
print("Nilai H dua gangguan:", nilai_H)
print("Amplitudo akhir dua orbit:", amplitudo_LV)
print(f"Waktu eksekusi sel komputasi = {waktu_notebook:.3f} s")

## Kesimpulan

Notebook ini menurunkan Soal 1 dari matriks stoikiometri dan memverifikasi dua hukum konservasi, menguji titik tetap serta matriks Jacobi Brusselator umum, menempatkan Hopf $A=1$ di $B=2$, dan membuktikan bahwa $B=4$ adalah batas diskriminan fokus–simpul. Simulasi $B=1{,}5,2,2{,}5,3,4,5,6$ menunjukkan peluruhan, ambang kritis, siklus limit, dan osilasi relaksasi $B=6$ melalui ukuran amplitudo serta penampang Poincaré.

Untuk Oregonator tereduksi, titik tetap positif diperoleh secara eksak, matriks Jacobi diperiksa dengan beda hingga, dan persamaan jejak menghasilkan $q_H\approx0{,}0540135346$. LSODA membedakan fokus stabil pada $q=0{,}1$ dari siklus limit menarik pada $q=0{,}01$. Terakhir, invarian Lotka–Volterra menunjukkan mengapa dua gangguan tetap berada pada orbit periodik berbeda, bukan tertarik menuju satu siklus limit yang sama.